# 🩺 Medical RAG Assistant
### Retrieval-Augmented Generation for Clinical Documents

---

**Project Summary:**  
This notebook demonstrates a complete **RAG (Retrieval-Augmented Generation)** pipeline applied to medical PDFs.  
Upload any clinical document → ask natural language questions → get grounded answers with source citations.

**Core Concepts Demonstrated:**
- PDF loading & text chunking
- Semantic embeddings (local HuggingFace model — no paid API)
- Vector store creation & similarity search (ChromaDB)
- LLM prompting with retrieved context (Google Gemini)
- Source-cited answer generation

**Tech Stack:**
| Component | Library |
|---|---|
| PDF Parsing | `pypdf` |
| Text Chunking | `langchain` RecursiveCharacterTextSplitter |
| Embeddings | `sentence-transformers` (all-MiniLM-L6-v2, CPU) |
| Vector DB | `chromadb` |
| LLM | Google Gemini 2.5 Flash (via `langchain-google-genai`) |
| Orchestration | `langchain` |

## ⚙️ Step 0 — Install Dependencies

Run this cell once if you are on a fresh environment (e.g., Google Colab or a new virtualenv).

In [1]:
# Uncomment and run if packages are not already installed
# !pip install langchain langchain-community langchain-google-genai chromadb sentence-transformers pypdf python-dotenv -q

## 📦 Step 1 — Imports & Configuration

In [2]:
import os
import time
import shutil

from dotenv import load_dotenv

# LangChain document loaders & splitters
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embeddings — local HuggingFace model (no API cost)
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector store
from langchain_community.vectorstores import Chroma

# LLM — Google Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# Load API key from .env file
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not GEMINI_API_KEY:
    raise EnvironmentError(
        "❌ GEMINI_API_KEY not found.\n"
        "Create a .env file in this directory with:\n"
        "  GEMINI_API_KEY=AIzaSy..."
    )

print("✅ API key loaded.")
print(f"   Key preview: {GEMINI_API_KEY[:8]}...{GEMINI_API_KEY[-4:]}")

C:\Users\lohit\AppData\Local\Temp\ipykernel_564\1794097940.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\Edu\Project\Medical RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain.text_splitter'

## 📄 Step 2 — Load & Chunk the Medical PDF

**Why chunking?**  
LLMs have a limited context window. Long documents must be split into small, overlapping chunks so the relevant portion can be retrieved for each query.

- `chunk_size=1000` → ~1000 characters per chunk
- `chunk_overlap=200` → 200-character overlap between consecutive chunks (preserves context at boundaries)

In [ ]:
# ── CONFIG: Change this to your PDF file path ───────────────────────────────
PDF_PATH = "data/sample_medical.pdf"  # ← Put your PDF here
# ────────────────────────────────────────────────────────────────────────────

def load_and_split_pdf(pdf_path: str, chunk_size: int = 1000, chunk_overlap: int = 200):
    """
    Load a PDF file and split it into overlapping text chunks.
    
    Args:
        pdf_path:     Path to the PDF file.
        chunk_size:   Max characters per chunk.
        chunk_overlap: Characters of overlap between chunks.
    
    Returns:
        chunks: List of LangChain Document objects (each = one chunk).
        pages:  List of raw page Documents (before splitting).
    """
    print(f"📖 Loading PDF: {pdf_path}")
    
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()                        # One Document per PDF page
    
    print(f"   → {len(pages)} pages loaded")
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = splitter.split_documents(pages)     # Split pages into chunks
    
    print(f"   → {len(chunks)} chunks created (size≈{chunk_size}, overlap={chunk_overlap})")
    return chunks, pages


# Run it!
chunks, pages = load_and_split_pdf(PDF_PATH)

# Preview the first chunk
print("\n--- First Chunk Preview ---")
print(f"Source: {chunks[0].metadata.get('source', 'N/A')}")
print(f"Page:   {chunks[0].metadata.get('page', 'N/A') + 1}")
print(f"Text:   {chunks[0].page_content[:300]}...")

## 🔢 Step 3 — Create Embeddings & Build Vector Store

**What is an embedding?**  
An embedding converts text into a dense numeric vector that captures semantic meaning.  
Similar sentences have vectors that are close together in vector space.

**Model used:** `all-MiniLM-L6-v2`
- 384-dimensional vectors
- Runs on CPU — no GPU or API key needed
- Fast and accurate for semantic search

**Vector store:** ChromaDB (local, persisted to disk)

In [ ]:
VECTOR_DB_DIR = "./chroma_db"   # Local directory to persist the vector store
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

def build_vector_store(chunks, persist_dir: str = VECTOR_DB_DIR):
    """
    Embed all chunks and store them in a ChromaDB vector store.
    
    Steps:
      1. Load a local HuggingFace embedding model.
      2. Convert each chunk into a 384-dim embedding vector.
      3. Store all vectors in ChromaDB (persisted locally).
    
    Returns:
        vectorstore: Chroma object ready for similarity search.
    """
    # Clean old DB if exists
    if os.path.exists(persist_dir):
        shutil.rmtree(persist_dir)
        print(f"🗑️  Cleared existing vector store at {persist_dir}")
    
    print(f"⚙️  Loading embedding model: {EMBEDDING_MODEL}")
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},     # CPU inference — no GPU needed
        encode_kwargs={"normalize_embeddings": True},   # Cosine similarity
    )
    print("   → Embedding model loaded")
    
    print(f"📐 Embedding {len(chunks)} chunks into ChromaDB...")
    start = time.time()
    
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_dir,
    )
    
    elapsed = time.time() - start
    print(f"✅ Vector store built in {elapsed:.1f}s")
    print(f"   → {vectorstore._collection.count()} vectors stored at '{persist_dir}'")
    
    return vectorstore


vectorstore = build_vector_store(chunks)

## 🔍 Step 4 — Set Up the Retriever

The retriever performs **semantic similarity search** — given a user query, it finds the top-K most relevant chunks from the vector store.

- Uses cosine similarity between query embedding and stored chunk embeddings
- `k=3` → retrieves the 3 most relevant chunks per query

In [ ]:
# Number of chunks to retrieve per query
TOP_K = 3

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K},
)

print(f"✅ Retriever configured — will fetch top {TOP_K} relevant chunks per query")

# Quick test: manually retrieve chunks for a sample question
test_query = "What are the main symptoms described?"
test_docs = retriever.invoke(test_query)

print(f"\n🔍 Test retrieval for: '{test_query}'")
for i, doc in enumerate(test_docs, 1):
    page = doc.metadata.get('page', 0) + 1
    src  = os.path.basename(doc.metadata.get('source', 'unknown'))
    print(f"\n  [{i}] {src} — Page {page}")
    print(f"       {doc.page_content[:200]}...")

## 🤖 Step 5 — Initialize the LLM (Google Gemini)

**Why Gemini?**
- Gemini 2.5 Flash is fast, accurate, and has a large context window
- `temperature=0.1` → near-deterministic output, minimizes hallucinations for medical QA

**Why low temperature?**  
In clinical settings, accuracy is critical. A low temperature means the model sticks closely to the retrieved context instead of "getting creative".

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.1,     # Low = factual, High = creative
)

print("✅ Gemini 2.5 Flash initialized")
print(f"   Model: gemini-2.5-flash")
print(f"   Temperature: 0.1 (low hallucination risk)")

## 📝 Step 6 — Define the RAG Prompt Template

This is the **prompt engineering** step — arguably the most important for quality results.

The prompt:
1. Gives the LLM a role: "expert Medical AI Assistant"
2. Injects the retrieved chunks as `{context}`
3. Instructs strict grounding — no hallucinations
4. Handles the edge case: if the document doesn't contain the answer

In [ ]:
SYSTEM_PROMPT = """You are an expert Medical AI Assistant. Answer questions accurately \
using ONLY the provided context. Follow these rules strictly:
1. Base your answer entirely on the context below. Do not use outside knowledge.
2. Be concise, professional, and clinically precise.
3. If the context does not contain enough information to answer, respond with:
   'I cannot answer this based on the provided document.'
4. Never hallucinate or speculate. Every statement must be grounded in the source text.

--- Context ---
{context}
---------------"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

print("✅ RAG prompt template configured")
print("   Rules enforced:")
print("   • Answers grounded ONLY in retrieved context")
print("   • No hallucination")
print("   • Graceful fallback if context is insufficient")

## 🔗 Step 7 — The Full RAG Pipeline

This function ties everything together:

```
User Query
    ↓
[Retriever]  →  Top-3 relevant chunks from ChromaDB
    ↓
[Prompt]     →  Inject chunks as context + user question
    ↓
[LLM]        →  Generate grounded answer
    ↓
Answer + Source Citations
```

In [ ]:
def run_rag_pipeline(query: str, retriever, llm, prompt_template) -> dict:
    """
    Execute the full Retrieval-Augmented Generation pipeline.
    
    Args:
        query:           User's natural language question.
        retriever:       ChromaDB-backed LangChain retriever.
        llm:             Initialized Gemini LLM.
        prompt_template: ChatPromptTemplate with system + user messages.
    
    Returns:
        dict:
            'answer'           → str   : LLM-generated grounded response
            'source_documents' → list  : Retrieved chunks used as context
            'elapsed_seconds'  → float : Time taken end-to-end
    """
    start = time.time()
    
    # ── Step A: Retrieve relevant chunks ───────────────────────────────────
    source_docs = retriever.invoke(query)
    
    # ── Step B: Format chunks into a context block ──────────────────────────
    context_text = "\n\n".join([
        f"[Page {doc.metadata.get('page', 0) + 1} | {os.path.basename(doc.metadata.get('source', 'Unknown'))}]\n{doc.page_content}"
        for doc in source_docs
    ])
    
    # ── Step C: Build prompt messages ───────────────────────────────────────
    messages = prompt_template.format_messages(
        context=context_text,
        question=query,
    )
    
    # ── Step D: Call the LLM ────────────────────────────────────────────────
    response = llm.invoke(messages)
    
    elapsed = time.time() - start
    
    return {
        "answer": response.content,
        "source_documents": source_docs,
        "elapsed_seconds": elapsed,
    }


print("✅ RAG pipeline function defined")

## ❓ Step 8 — Ask Questions!

Now we can query the medical document. Change `QUERY` to any clinical question about your uploaded PDF.

In [ ]:
# ── Change this to your question ────────────────────────────────────────────
QUERY = "What are the diagnostic criteria mentioned in the document?"
# ────────────────────────────────────────────────────────────────────────────

print(f"❓ Query: {QUERY}\n")
print("⏳ Running RAG pipeline...\n")

result = run_rag_pipeline(
    query=QUERY,
    retriever=retriever,
    llm=llm,
    prompt_template=prompt_template,
)

# ── Display Answer ──────────────────────────────────────────────────────────
print("=" * 60)
print("💡 GROUNDED ANSWER")
print("=" * 60)
print(result["answer"])
print(f"\n⏱️  Generated in {result['elapsed_seconds']:.2f}s")

## 📚 Step 9 — View Source Citations

A critical feature of RAG: **every answer is traceable to specific pages in the source document**.  
This is what makes it trustworthy for medical use — the interviewer can verify each claim.

In [ ]:
print("=" * 60)
print("📚 SOURCE CITATIONS")
print("=" * 60)

seen = set()   # Deduplicate identical chunks
for i, doc in enumerate(result["source_documents"], 1):
    chunk_text = doc.page_content.strip()
    if chunk_text in seen:
        continue
    seen.add(chunk_text)
    
    file_name = os.path.basename(doc.metadata.get("source", "Unknown"))
    page_num  = doc.metadata.get("page", 0) + 1
    preview   = chunk_text[:500] + ("..." if len(chunk_text) > 500 else "")
    
    print(f"\n[{i}] 📄 {file_name} — Page {page_num}")
    print("-" * 50)
    print(preview)

## 🔄 Step 10 — Interactive Multi-Turn Q&A Loop

Run this cell to enter a loop where you can ask multiple questions one after another.  
Type `quit` or `exit` to stop.

In [ ]:
print("🩺 Medical RAG Assistant — Interactive Mode")
print("   Type your question and press Enter.")
print("   Type 'quit' to exit.\n")
print("-" * 60)

while True:
    try:
        query = input("\n❓ Your question: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 Exiting.")
        break
    
    if not query:
        print("   ⚠️  Please enter a question.")
        continue
    
    if query.lower() in ("quit", "exit", "q"):
        print("👋 Goodbye!")
        break
    
    print("⏳ Thinking...\n")
    
    try:
        result = run_rag_pipeline(
            query=query,
            retriever=retriever,
            llm=llm,
            prompt_template=prompt_template,
        )
        
        print("=" * 60)
        print("💡 ANSWER")
        print("=" * 60)
        print(result["answer"])
        print(f"\n⏱️  {result['elapsed_seconds']:.2f}s")
        
        print("\n📚 Sources used:")
        seen = set()
        for doc in result["source_documents"]:
            key = doc.page_content.strip()
            if key not in seen:
                seen.add(key)
                fname = os.path.basename(doc.metadata.get("source", "?"))
                pg    = doc.metadata.get("page", 0) + 1
                print(f"   • {fname} — Page {pg}")
        
        print("-" * 60)
    
    except Exception as e:
        print(f"❌ Error: {e}")

## 🧠 Appendix — RAG Architecture Explained

```
                    ┌─────────────────────────────────┐
                    │        INDEXING PHASE            │
                    │  (done once, before questions)   │
                    └─────────────────────────────────┘
                                   │
        PDF File                   │
            │                      │
            ▼                      │
    ┌───────────────┐             │
    │  PyPDFLoader  │ Load pages  │
    └───────────────┘             │
            │                      │
            ▼                      │
    ┌─────────────────┐           │
    │  TextSplitter   │ Chunk text │
    └─────────────────┘           │
            │                      │
            ▼                      │
    ┌──────────────────────┐      │
    │  HuggingFace Embedder│ Embed│
    │  (all-MiniLM-L6-v2)  │      │
    └──────────────────────┘      │
            │                      │
            ▼                      │
    ┌───────────────────┐         │
    │    ChromaDB        │ Store   │
    │  (vector store)    │ ────────┘
    └───────────────────┘


                    ┌─────────────────────────────────┐
                    │        QUERY PHASE               │
                    │  (happens on every question)     │
                    └─────────────────────────────────┘

    User Query
        │
        ▼
    [Embed query]  ──→  384-dim vector
        │
        ▼
    [ChromaDB similarity search]  ──→  Top-3 similar chunks
        │
        ▼
    [Format prompt with context + question]
        │
        ▼
    [Gemini 2.5 Flash LLM]
        │
        ▼
    Grounded Answer + Page Citations
```

### Key Design Decisions

| Decision | Choice | Reason |
|---|---|---|
| Embedding model | `all-MiniLM-L6-v2` | Free, fast, CPU-only, good accuracy |
| Vector DB | ChromaDB | Simple, local, no server needed |
| LLM | Gemini 2.5 Flash | Fast, large context window, affordable |
| Temperature | 0.1 | Minimize hallucination for medical use |
| Chunk size | 1000 chars | Balance between context richness & retrieval precision |
| Top-K | 3 | Enough context without overwhelming the prompt |

### Limitations & Future Work
- **No memory**: Each question is independent (no multi-turn context)
- **PDF quality**: Scanned PDFs need OCR pre-processing
- **Re-ranking**: Could add a cross-encoder re-ranker after initial retrieval
- **Evaluation**: Could add RAGAS metrics (faithfulness, answer relevancy)